In [18]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
import torch.optim as optim
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup #hugging face
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from tqdm import tqdm
from datetime import datetime
import random

In [19]:
# 1. Definir o Dataset customizado para treino/validação
class TextDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128, is_train=True):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train

    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, index):
        text = self.dataframe.loc[index, 'ds_gastos']
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt'
        )
        item = {
            'input_ids': encoding['input_ids'].squeeze(),  # remove dimensao extra
            'attention_mask': encoding['attention_mask'].squeeze()
        }
        if self.is_train:
            label = self.dataframe.loc[index, 'y']
            item['labels'] = torch.tensor(label, dtype=torch.long)
        return item

# Dataset para aplicação (sem labels)
class AppDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_len=128):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, index):
        text = self.dataframe.loc[index, 'ds_gastos']
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            truncation=True,
            padding='max_length',
            return_attention_mask=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze()
        }

In [5]:
# 2. Carregar os dados e dividir em treino e validação
file_parquet = 'bases/bd01_gastos_treino.parquet'
df = pd.read_parquet(file_parquet)
#df = df.sample(n=1000, random_state=42)
df['y'] = df['y'].astype(int)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)

In [8]:
# 3. Carregar o tokenizer e o modelo BERT para português
model_name = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=14)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-large-portuguese-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
# use abaixo se precisar carregar um modelo já salvo
# 3. Carregar o tokenizer e o modelo BERT para português
model_name = "neuralmind/bert-large-portuguese-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained("modelo/md02_bert_final/")

In [30]:
# 4. Configurar o dispositivo (GPU com CUDA, se disponível)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
device

device(type='cuda')

In [10]:
# 5. Criar os datasets e dataloaders
max_len    = 93   # número máximo de tokens por texto
batch_size = 16

# Instancia os datasets
train_dataset = TextDataset(train_df, tokenizer, max_len=max_len, is_train=True)
val_dataset   = TextDataset(val_df,   tokenizer, max_len=max_len, is_train=True)

# Opção A: Para bases pequenas (usar todo o dataset por época)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,    # embaralha as 400 observações a cada época
    drop_last=False  # não descarta o último batch, mesmo que menor que `batch_size`
)
val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False    # normalmente não embaralha validação
)

# --------------------------------------------------------
# Opção B: Para bases MUITO grandes, amostrar apenas uma fração
# (descomente se quiser usar sampler em datasets maiores que sampler_size)

# sampler_size = 16000
# indices      = random.sample(range(len(train_dataset)), sampler_size)
# sampler      = SubsetRandomSampler(indices)
#
# train_loader = DataLoader(
#     train_dataset,
#     batch_size=batch_size,
#     sampler=sampler
# )
# --------------------------------------------------------


In [11]:
# 6. Configurar o otimizador e scheduler
epochs = 100
optimizer = optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(optimizer,
                                            num_warmup_steps=int(0.1 * total_steps),
                                            num_training_steps=total_steps)

In [15]:
# Imprime timestamp de início
print("Início do treino:", datetime.now())

patience = 3
best_val_loss = float('inf')
patience_counter = 0

# 7. Loop de treinamento
for epoch in range(1, epochs + 1):
    # --- Treino ---
    model.train()
    train_loss = 0.0
    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch}/{epochs}",
        leave=False
    )

    for batch in progress_bar:
        optimizer.zero_grad()
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        train_loss += loss.item()
        progress_bar.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss / len(train_loader)
    print(f"\nEpoch {epoch} — Treino Loss: {avg_train_loss:.4f}")

    # --- Validação ---
    model.eval()
    val_loss = 0.0
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            val_loss += outputs.loss.item()
            logits = outputs.logits

            preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
            true_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    acc = accuracy_score(true_labels, preds)
    precision, recall, f1, _ = precision_recall_fscore_support(
        true_labels, preds, average='weighted'
    )
    print(
        f"Epoch {epoch} — Val Loss: {avg_val_loss:.4f} | "
        f"Acc: {acc:.4f} | Prec: {precision:.4f} | "
        f"Rec: {recall:.4f} | F1: {f1:.4f}\n"
    )

    # --- Early Stopping ---
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        print(f"Sem melhora na loss de validação por {patience_counter} epoch(s).")

    if patience_counter >= patience:
        print(
            f"Early stopping ativado: nenhum ganho em {patience} epochs consecutivas."
        )
        break

# Imprime timestamp de término
print("Fim do treino:", datetime.now())

Início do treino: 2025-04-30 03:03:14.858019



Epoch 1 — Treino Loss: 2.5102


C:\Users\marco\miniconda3\envs\bertclass\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 1 — Val Loss: 2.5148 | Acc: 0.2500 | Prec: 0.0676 | Rec: 0.2500 | F1: 0.1064




Epoch 2 — Treino Loss: 2.3839


...


Epoch 15 — Treino Loss: 0.0668
Epoch 15 — Val Loss: 0.6587 | Acc: 0.8000 | Prec: 0.7832 | Rec: 0.8000 | F1: 0.7832

Sem melhora na loss de validação por 2 epoch(s).



Epoch 16 — Treino Loss: 0.0499
Epoch 16 — Val Loss: 0.6275 | Acc: 0.8250 | Prec: 0.8355 | Rec: 0.8250 | F1: 0.8210




Epoch 17 — Treino Loss: 0.0405
Epoch 17 — Val Loss: 0.6478 | Acc: 0.8250 | Prec: 0.8355 | Rec: 0.8250 | F1: 0.8210

Sem melhora na loss de validação por 1 epoch(s).



Epoch 18 — Treino Loss: 0.0344
Epoch 18 — Val Loss: 0.6665 | Acc: 0.8125 | Prec: 0.8269 | Rec: 0.8125 | F1: 0.8067

Sem melhora na loss de validação por 2 epoch(s).



Epoch 19 — Treino Loss: 0.0292
Epoch 19 — Val Loss: 0.6589 | Acc: 0.8125 | Prec: 0.8269 | Rec: 0.8125 | F1: 0.8067

Sem melhora na loss de validação por 3 epoch(s).
Early stopping ativado: nenhum ganho em 3 epochs consecutivas.
Fim do treino: 2025-04-30 03:05:51.639209


In [16]:

# Salvar o modelo treinado e o tokenizer
model.save_pretrained("modelo/md02_bert_final")
tokenizer.save_pretrained("modelo/md02_bert_final")

('modelo/md02_bert_final\\tokenizer_config.json',
 'modelo/md02_bert_final\\special_tokens_map.json',
 'modelo/md02_bert_final\\vocab.txt',
 'modelo/md02_bert_final\\added_tokens.json',
 'modelo/md02_bert_final\\tokenizer.json')

In [31]:
# 8. Aplicar o modelo na base de dados de aplicação
# Supondo que 'dados_aplicacao.csv' contenha a coluna "Y" (sem label)
app_df = pd.read_parquet("bases/bd01_gastos_apply.parquet")
#app_df = app_df.sample(n=5000, random_state=42)


In [32]:
app_dataset = AppDataset(app_df, tokenizer, max_len=max_len)
app_loader = DataLoader(app_dataset, batch_size=batch_size)

In [33]:
model.eval()
all_preds = []
with torch.no_grad():
    for batch in tqdm(app_loader, desc="Predizendo na base de aplicação"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)
        all_preds.extend(preds.cpu().numpy())

app_df['y'] = all_preds
app_df.to_parquet("output/base_final.parquet", index=False)
print("Predições salvas'")


Predizendo na base de aplicação: 100%|█████████████████████████████████████████████████| 63/63 [00:06<00:00,  9.97it/s]


Predições salvas'
